# 02 Split and baseline

**Stage 1.** Freeze the train and test split, cut the features, build the response baseline and run the first uplift check for Gate G1.

**Why a fixed holdout.** Conversion is rare. A 30 percent test slice of a 1M sample holds only about 90 control conversions, too few for a stable Qini curve. So we hold out 2M rows from the full 14M row file, once, by seed. Nothing is ever fitted, tuned or selected on them. Training samples of 100k, 500k, 1M and 5M rows are nested prefixes of one shuffled pool, so a bigger sample always contains the smaller one and the data size curve compares like with like.

In [1]:
import json, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
RESULTS, REPORT = ROOT / "results", ROOT / "report"
RESULTS.mkdir(exist_ok=True); REPORT.mkdir(exist_ok=True)

from uplift.data import FEATURES, TREATMENT, OUTCOMES, make_splits, take, treated_share_ok, TRAIN_SIZES
from uplift import metrics as m

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
NAMES = {"response": "Response baseline", "s": "S-learner", "t": "T-learner", "x": "X-learner"}
COLORS = {"response": "#7a7a7a", "s": "#1c7293", "t": "#f2a93b", "x": "#0b3d5c"}
df = pd.read_parquet(ROOT / "data" / "criteo.parquet")
test_ids, pool_ids = make_splits(len(df))
assert len(np.intersect1d(test_ids, pool_ids[:5_000_000])) == 0, "test rows leaked into training"
train = take(df, pool_ids[:1_000_000])   # development sample
test = take(df, test_ids)                 # frozen holdout
print(f"full file {len(df):,} | holdout {len(test):,} | development train {len(train):,}")
print(f"treated share: train {train[TREATMENT].mean():.4f}, test {test[TREATMENT].mean():.4f} (file is ordered, so we check)")
assert treated_share_ok(train) and treated_share_ok(test)
rates = pd.concat({"train": train.groupby(TREATMENT)[OUTCOMES].mean(), "test": test.groupby(TREATMENT)[OUTCOMES].mean()})
rates

full file 13,979,592 | holdout 2,000,000 | development train 1,000,000
treated share: train 0.8503, test 0.8497 (file is ordered, so we check)


visit  conversion
      treatment                      
train 0          0.037587    0.001997
      1          0.048864    0.003126
test  0          0.038549    0.001956
      1          0.048501    0.003129

## Feature cut, 12 to 10

The guideline allows about 10 features. We drop two, using training rows only. We fit a response model and drop the two features with the lowest importance. For each one we also record its strongest correlation with a feature we keep, to see whether its information is duplicated elsewhere. We checked on a validation slice inside the training data that dropping these two costs almost nothing in AUC.

In [2]:
from uplift.pipeline import select_features
kept, log, importance = select_features(train, FEATURES)
print("kept:", kept)
display(log.round(3))
fs = importance.rename("importance").to_frame().assign(kept=lambda d: d.index.isin(kept))
fs.round(4).to_csv(RESULTS / "feature_selection.csv")
json.dump(kept, open(RESULTS / "kept_features.json", "w"))
fs.sort_values("importance", ascending=False).round(4)

kept: ['f0', 'f2', 'f3', 'f4', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']


,dropped,importance,most_similar_kept,abs_corr_with_it
0,f1,0.001,f5,0.575
1,f5,0.001,f7,0.745


,importance,kept
f8,0.5682,True
f9,0.2205,True
f2,0.1485,True
f6,0.0207,True
f0,0.0151,True
f3,0.0106,True
f10,0.0054,True
f7,0.0044,True
f4,0.0035,True
f11,0.0016,True


## Response baseline

A response model predicts who is likely to have the outcome and ignores treatment. It is the benchmark every uplift model has to beat. We fit logistic regression and LightGBM on the development sample and score the holdout. AUC measures ranking of the outcome. Precision and recall are read at the top 20 percent of scores.

In [3]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from uplift.learners import BASE_PARAMS

rows = []
for outcome in OUTCOMES:
    y_tr, y_te = train[outcome], test[outcome].to_numpy()
    models = {
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=300)),
        "LightGBM": LGBMClassifier(**BASE_PARAMS),
    }
    for name, model in models.items():
        p = model.fit(train[kept], y_tr).predict_proba(test[kept])[:, 1]
        top = p >= np.quantile(p, 0.8)
        rows.append(dict(outcome=outcome, model=name, auc=roc_auc_score(y_te, p),
                         precision_top20=y_te[top].mean(), recall_top20=y_te[top].sum() / y_te.sum()))
baseline = pd.DataFrame(rows)
baseline.round(4).to_csv(RESULTS / "baseline_metrics.csv", index=False)
baseline.round(4)

,outcome,model,auc,precision_top20,recall_top20
0,visit,Logistic regression,0.9312,0.2137,0.9092
1,visit,LightGBM,0.9466,0.2174,0.9250
2,conversion,Logistic regression,0.9508,0.0139,0.9423
3,conversion,LightGBM,0.9571,0.0140,0.9494


## Gate G1 check

Two conditions: the average effect is positive, and a quick T-learner beats random targeting on visits. The balance table and average effects come from notebook 01. This cell repeats the average effect on the holdout and runs the quick T-learner.

In [4]:
from uplift.learners import TLearner
y, t = test["visit"].to_numpy(), test[TREATMENT].to_numpy()
ate = y[t == 1].mean() - y[t == 0].mean()
quick = TLearner().fit(train[kept], train[TREATMENT], train["visit"]).predict_uplift(test[kept])
coef = m.qini_coefficient(y, t, quick)
rand = np.random.default_rng(0).random(len(y))
boot = m.bootstrap_metrics(y, t, {"quick": quick, "random": rand}, n_boot=100)
d = m.paired_difference(boot, "quick", "random", coef, m.qini_coefficient(y, t, rand))
print(f"average effect on visits, holdout: {ate:.4f}")
print(f"quick T-learner Qini coefficient: {coef:.5f} | beats random: difference {d['diff']:.5f}, 95% CI [{d['lo']:.5f}, {d['hi']:.5f}], p = {d['p_one_sided']:.3f}")
print("G1:", "PASS" if ate > 0 and d["lo"] > 0 else "FAIL")

average effect on visits, holdout: 0.0100
quick T-learner Qini coefficient: 0.00313 | beats random: difference 0.00324, 95% CI [0.00283, 0.00354], p = 0.010
G1: PASS
